# 데이터 사이언스를 위한 생성형 인공지능
## 1주차 실습 — Multimodal Generation Mini Lab

**동덕여자대학교 데이터사이언스전공 유원상 교수 · 2026년 2학기**

> 학생용 노트북: TODO와 비교 실험을 직접 수행하고 결과를 자신의 말로 해석합니다.

---

### 해당 수업의 주제

- 강의 소개와 실습 환경 확인
- 생성형 AI의 공통 인터페이스: `prompt + model + randomness → sample`
- seed, 확률적 샘플링, temperature
- 텍스트·이미지·오디오 데이터의 생성 체험
- 책임 있는 생성형 AI 점검

### 수업 개요

이 노트북은 교재 1장과 제공된 `01_introduction.ipynb`의 이미지·텍스트·오디오 생성 흐름을 첫 수업에 맞게 경량화한 실습입니다. 핵심 실습은 CPU에서도 실행되며, 선택 실습에서만 사전학습 텍스트 모델을 다운로드합니다.

> 이미지·오디오의 기본 실습은 딥러닝 모델이 아니라 생성의 공통 원리와 데이터 표현을 이해하기 위한 **교육용 확률·절차적 기준선**입니다. 이후 주차에서 Transformer, Diffusion, Stable Diffusion, MusicGen으로 교체합니다.


## 0. Colab 실습환경 설정

### 체크리스트

1. **파일 → Drive에 사본 저장**으로 개인 사본을 만듭니다.
2. 아래 환경 점검 셀을 실행합니다.
3. 본인의 이름과 seed를 입력합니다.
4. 출력이 다르거나 오류가 발생하면 메시지를 지우지 말고 원인을 기록합니다.
5. API 키가 필요한 실습에서도 키를 코드에 직접 입력하지 않습니다.


In [ ]:
# 환경 점검: 이 셀은 설치 없이 실행되어야 합니다.
import os
import sys
import platform
import random
from importlib import metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display

try:
    import torch
except Exception as exc:
    torch = None
    print("[주의] PyTorch import 실패:", exc)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", metadata.version("matplotlib"))
print("PyTorch:", getattr(torch, "__version__", "not available"))


### 실습 0-1. 개인 설정

- `STUDENT_NAME`을 본인 이름으로 바꾸십시오.
- `SEED`는 학번 끝 두 자리 등을 이용해 본인만의 정수로 바꾸어도 좋습니다.
- 선택 사전학습 모델은 기본값 `False`로 둡니다. 인터넷 연결과 수업 시간이 허용될 때만 `True`로 바꿉니다.


In [ ]:
STUDENT_NAME = "이름을 입력하세요"
SEED = 42
RUN_OPTIONAL_PRETRAINED_DEMO = False

print(f"실습자: {STUDENT_NAME} | seed: {SEED} | optional model: {RUN_OPTIONAL_PRETRAINED_DEMO}")


In [ ]:
def set_global_seed(seed: int) -> None:
    """Python, NumPy, PyTorch의 난수 초기값을 가능한 범위에서 고정합니다."""
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    if torch is not None and torch.cuda.is_available():
        return "cuda"
    if torch is not None and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

set_global_seed(SEED)
DEVICE = get_device()
print("Using device:", DEVICE)
if torch is not None and DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


#### 확인 질문

1. `DEVICE`가 `cpu`여도 오늘의 핵심 실습을 수행할 수 있습니까?
2. seed를 고정하면 어떤 조건에서 같은 결과를 얻을 가능성이 높아집니까?
3. seed가 같아도 패키지 버전·하드웨어·연산 방식이 크게 달라지면 결과가 완전히 같지 않을 수 있는 이유를 생각해 봅시다.


## 1. 확률분포에서 “생성”하기

생성은 가장 단순하게 말하면 **가능한 출력들의 확률분포에서 하나를 선택하는 것**입니다. 다음 예에서는 네 개의 토큰 후보 중 하나를 샘플링합니다.


In [ ]:
tokens = np.array(["AI", "데이터", "모델", "창의성"])
base_probs = np.array([0.50, 0.25, 0.15, 0.10], dtype=float)

rng = np.random.default_rng(SEED)
samples = rng.choice(tokens, size=20, p=base_probs)
print("생성 샘플:", " | ".join(samples))

plt.figure(figsize=(7, 3.2))
plt.bar(tokens, base_probs)
plt.ylim(0, 0.6)
plt.ylabel("probability")
plt.title("A tiny categorical generative model")
plt.show()


### 실습 1-1. Temperature 함수 완성하기

확률 `p_i`에 temperature `T`를 적용하는 한 방법은 다음과 같습니다.

1. `log(p_i)`를 계산합니다.
2. `T`로 나눕니다.
3. 지수화한 뒤 합이 1이 되도록 정규화합니다.

아래 학생 함수는 현재 temperature를 무시하는 **임시 기준선**입니다. TODO 부분을 수정하여 `T=0.5`, `1.0`, `2.0`에서 서로 다른 분포가 나오도록 만드십시오.


In [ ]:
def apply_temperature_student(probs, temperature):
    probs = np.asarray(probs, dtype=float)
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    # TODO: log 확률을 temperature로 조정하고 다시 정규화하십시오.
    # 현재 코드는 단순 정규화만 하므로 temperature가 결과에 영향을 주지 않습니다.
    adjusted = probs.copy()
    return adjusted / adjusted.sum()

for t in [0.5, 1.0, 2.0]:
    print(f"T={t}:", np.round(apply_temperature_student(base_probs, t), 3))


### 실습 1-2. 일부러 오류 만들고 고치기

`numpy.random.Generator.choice`에 전달하는 확률의 합은 1이어야 합니다. 아래 셀에서 `RUN_DEBUG_EXERCISE=True`로 바꾸면 오류가 발생합니다.

1. 오류 메시지에서 핵심 문장을 찾습니다.
2. `bad_probs`의 합을 출력합니다.
3. 합이 1이 되도록 정규화한 뒤 다시 실행합니다.


In [ ]:
RUN_DEBUG_EXERCISE = False

if RUN_DEBUG_EXERCISE:
    bad_probs = np.array([0.20, 0.30, 0.60])  # 합이 1.10입니다.
    print("sum =", bad_probs.sum())
    debug_rng = np.random.default_rng(SEED)
    print(debug_rng.choice(["A", "B", "C"], size=5, p=bad_probs))
else:
    print("디버깅할 때 RUN_DEBUG_EXERCISE를 True로 바꾸십시오.")


## 2. 문자 수준 텍스트 생성

현대 언어 모델은 토큰 단위의 복잡한 확률분포를 학습하지만, 핵심 아이디어는 “현재 문맥 다음에 올 가능성이 높은 단위를 샘플링한다”는 것입니다. 여기서는 작은 문장에서 **문자 bigram 모델**을 만듭니다.


In [ ]:
CORPUS = (
    "생성형 인공지능은 데이터에서 패턴을 배우고 새로운 결과를 만든다. "
    "같은 프롬프트라도 시드와 샘플링 설정에 따라 결과가 달라질 수 있다. "
    "좋은 결과는 자연스러움뿐 아니라 사실성 공정성 안전성을 함께 살펴야 한다. "
    "데이터 사이언티스트는 모델의 출력과 한계를 검증하고 기록한다. "
) * 8


def build_bigram_model(text: str):
    vocab = sorted(set(text))
    idx = {ch: i for i, ch in enumerate(vocab)}
    counts = np.ones((len(vocab), len(vocab)), dtype=float)  # Laplace smoothing
    for a, b in zip(text[:-1], text[1:]):
        counts[idx[a], idx[b]] += 1
    probs = counts / counts.sum(axis=1, keepdims=True)
    return vocab, idx, probs

VOCAB, CHAR_TO_IDX, BIGRAM_PROBS = build_bigram_model(CORPUS)
print("문자 종류:", len(VOCAB), "| 전이행렬 shape:", BIGRAM_PROBS.shape)


In [ ]:
def apply_temperature(probs, temperature):
    probs = np.asarray(probs, dtype=float)
    logits = np.log(probs + 1e-12)
    scaled = np.exp(logits / temperature)
    return scaled / scaled.sum()


def generate_text_reference(start="생", length=120, temperature=1.0, seed=42):
    rng = np.random.default_rng(seed)
    current = start if start in CHAR_TO_IDX else VOCAB[0]
    out = [current]
    for _ in range(length - 1):
        probs = BIGRAM_PROBS[CHAR_TO_IDX[current]]
        probs = apply_temperature(probs, temperature)
        current = rng.choice(VOCAB, p=probs)
        out.append(current)
    return "".join(out)

for t in [0.55, 1.0, 1.8]:
    print(f"\n[T={t}]\n{generate_text_reference(temperature=t, seed=SEED)}")


### 실습 2-1. Greedy 생성기를 확률적 생성기로 바꾸기

아래 함수는 다음 문자를 항상 가장 높은 확률의 문자로 고르는 greedy 기준선입니다.

- TODO 1: `apply_temperature`를 사용하십시오.
- TODO 2: `np.argmax` 대신 `rng.choice(..., p=...)`를 사용하십시오.
- 서로 다른 temperature와 seed로 결과를 비교하십시오.


In [ ]:
def generate_text_student(start="생", length=100, temperature=1.0, seed=42):
    rng = np.random.default_rng(seed)
    current = start if start in CHAR_TO_IDX else VOCAB[0]
    out = [current]

    for _ in range(length - 1):
        probs = BIGRAM_PROBS[CHAR_TO_IDX[current]]

        # TODO 1: temperature를 적용한 확률로 바꾸십시오.
        adjusted_probs = probs

        # TODO 2: 확률적 샘플링으로 바꾸십시오. 현재는 greedy 선택입니다.
        next_index = int(np.argmax(adjusted_probs))
        current = VOCAB[next_index]
        out.append(current)

    return "".join(out)

print(generate_text_student(temperature=1.2, seed=SEED))


### 실습 2-2. 다양성을 데이터로 비교하기

생성 결과의 “느낌”만 말하지 말고 간단한 지표를 계산합니다. 여기서는 서로 다른 문자 비율과 반복 bigram 비율을 사용합니다.


In [ ]:
def diversity_metrics(text: str):
    chars = [c for c in text if not c.isspace()]
    bigrams = list(zip(chars[:-1], chars[1:]))
    unique_char_ratio = len(set(chars)) / max(1, len(chars))
    repeated_bigram_ratio = 1 - len(set(bigrams)) / max(1, len(bigrams))
    return unique_char_ratio, repeated_bigram_ratio

rows = []
for t in [0.4, 0.7, 1.0, 1.4, 2.0]:
    txt = generate_text_reference(length=220, temperature=t, seed=SEED)
    ucr, rbr = diversity_metrics(txt)
    rows.append({"temperature": t, "unique_char_ratio": ucr, "repeated_bigram_ratio": rbr, "preview": txt[:45]})

diversity_df = pd.DataFrame(rows)
display(diversity_df)

plt.figure(figsize=(7, 3.4))
plt.plot(diversity_df["temperature"], diversity_df["unique_char_ratio"], marker="o", label="unique char ratio")
plt.plot(diversity_df["temperature"], diversity_df["repeated_bigram_ratio"], marker="o", label="repeated bigram ratio")
plt.xlabel("temperature")
plt.ylabel("ratio")
plt.ylim(0, 1)
plt.legend()
plt.title("Temperature changes the sampling behavior")
plt.show()


#### 해석 기록

아래 문장을 완성하십시오.

- temperature가 낮을수록 ________________________________________________.
- temperature가 높을수록 ________________________________________________.
- 다양성이 높다고 해서 품질과 사실성이 반드시 좋아지는 것은 아닌 이유는 ________________________________.


## 3. Prompt와 seed로 이미지 생성 체험

다음 함수는 신경망 이미지 생성기가 아닙니다. prompt의 키워드를 색·도형 매개변수로 바꾸고 seed로 무작위 구성을 만드는 **절차적 생성 기준선**입니다.

이 실습의 목적은 다음을 관찰하는 것입니다.

- 같은 prompt + 같은 seed → 같은 결과
- 같은 prompt + 다른 seed → 다른 구성
- prompt 변경 → 조건에 따른 색·형태 변화


In [ ]:
from matplotlib.patches import Circle, Rectangle, Polygon

STYLE_TABLE = {
    "calm": {"cmap": "Blues", "shape": "circle"},
    "bright": {"cmap": "YlOrRd", "shape": "circle"},
    "forest": {"cmap": "Greens", "shape": "rectangle"},
    "night": {"cmap": "Purples", "shape": "polygon"},
    "data": {"cmap": "viridis", "shape": "rectangle"},
}

def keyword_style(prompt: str):
    p = prompt.lower()
    for key, style in STYLE_TABLE.items():
        if key in p:
            return style
    return {"cmap": "plasma", "shape": "polygon"}

def generate_abstract_image(prompt: str, seed: int = 42, n_shapes: int = 32):
    rng = np.random.default_rng(seed)
    style = keyword_style(prompt)
    cmap = plt.get_cmap(style["cmap"])

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.set_title(f'prompt="{prompt}" | seed={seed}', fontsize=10)

    for i in range(n_shapes):
        x, y = rng.uniform(0, 1, size=2)
        size = rng.uniform(0.03, 0.18)
        color = cmap(rng.uniform(0.15, 0.95))
        alpha = rng.uniform(0.35, 0.85)
        if style["shape"] == "circle":
            patch = Circle((x, y), size/2, color=color, alpha=alpha)
        elif style["shape"] == "rectangle":
            patch = Rectangle((x-size/2, y-size/2), size, size*rng.uniform(0.5, 1.5), angle=rng.uniform(0, 180), color=color, alpha=alpha)
        else:
            pts = np.column_stack([x + rng.normal(0, size, 5), y + rng.normal(0, size, 5)])
            patch = Polygon(pts, closed=True, color=color, alpha=alpha)
        ax.add_patch(patch)
    return fig

fig = generate_abstract_image("calm data landscape", seed=SEED)
plt.show()


### 실습 3-1. 조건과 무작위성 분리하기

아래 변수를 바꾸어 3×2 비교를 만드십시오.

- prompt 2개
- 각 prompt마다 seed 3개

관찰 결과를 “내용 조건의 변화”와 “무작위 구성의 변화”로 구분해 설명하십시오.


In [ ]:
PROMPTS = ["calm data landscape", "bright data landscape"]  # TODO: 하나 이상 수정
SEEDS = [SEED, SEED + 1, SEED + 2]

fig, axes = plt.subplots(len(PROMPTS), len(SEEDS), figsize=(12, 7))
for r, prompt in enumerate(PROMPTS):
    for c, seed in enumerate(SEEDS):
        rng = np.random.default_rng(seed)
        style = keyword_style(prompt)
        cmap = plt.get_cmap(style["cmap"])
        ax = axes[r, c]
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
        ax.set_title(f"{prompt}\nseed={seed}", fontsize=9)
        for i in range(24):
            x, y = rng.uniform(0, 1, size=2)
            s = rng.uniform(0.03, 0.16)
            ax.add_patch(Circle((x, y), s/2, color=cmap(rng.uniform()), alpha=rng.uniform(0.35, 0.8)))
plt.tight_layout()
plt.show()


## 4. 오디오를 데이터로 보고 생성하기

오디오는 일정한 sampling rate로 기록된 진폭 배열입니다. 다음 교육용 생성기는 prompt 키워드에 따라 기본 주파수와 리듬을 정하고, seed로 작은 변화를 더합니다.


In [ ]:
def audio_style(prompt: str):
    p = prompt.lower()
    if "calm" in p or "차분" in p:
        return {"freqs": [220, 330], "tempo": 1.0}
    if "bright" in p or "밝" in p:
        return {"freqs": [440, 660], "tempo": 2.0}
    if "tense" in p or "긴장" in p:
        return {"freqs": [185, 277, 415], "tempo": 4.0}
    return {"freqs": [261.6, 392.0], "tempo": 1.5}

def generate_audio(prompt: str, seed: int = 42, duration: float = 2.5, sampling_rate: int = 16000):
    rng = np.random.default_rng(seed)
    style = audio_style(prompt)
    t = np.arange(int(duration * sampling_rate)) / sampling_rate
    y = np.zeros_like(t, dtype=float)

    for i, f in enumerate(style["freqs"]):
        phase = rng.uniform(0, 2*np.pi)
        y += (0.55 / (i+1)) * np.sin(2*np.pi*(f + rng.normal(0, 1.5))*t + phase)

    envelope = 0.5 * (1 - np.cos(2*np.pi*np.clip(t / duration, 0, 1)))
    pulse = 0.65 + 0.35 * (np.sin(2*np.pi*style["tempo"]*t) > 0)
    y = y * envelope * pulse
    y = y / (np.max(np.abs(y)) + 1e-9)
    return y.astype(np.float32), sampling_rate

AUDIO_PROMPT = "calm data sonification"
waveform, sampling_rate = generate_audio(AUDIO_PROMPT, seed=SEED)
print("waveform shape:", waveform.shape, "| sampling_rate:", sampling_rate, "| seconds:", len(waveform)/sampling_rate)
display(Audio(waveform, rate=sampling_rate))


In [ ]:
time_axis = np.arange(len(waveform)) / sampling_rate

plt.figure(figsize=(10, 3))
plt.plot(time_axis[:2000], waveform[:2000])
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title("Waveform: first 0.125 seconds")
plt.show()

plt.figure(figsize=(10, 3.5))
plt.specgram(waveform, NFFT=512, Fs=sampling_rate, noverlap=256)
plt.ylim(0, 1200)
plt.xlabel("time (s)")
plt.ylabel("frequency (Hz)")
plt.title("Spectrogram")
plt.colorbar(label="intensity")
plt.show()


### 실습 4-1. 오디오 제어변수 실험

1. `AUDIO_PROMPT`를 `bright`, `tense`, 또는 한글 키워드로 바꿉니다.
2. seed를 바꾸고 파형과 소리를 비교합니다.
3. sampling rate를 8,000과 16,000으로 바꾸어 배열 길이를 비교합니다.
4. “sampling rate를 절반으로 줄이면 같은 duration에서 배열 길이는 어떻게 되는가?”를 답합니다.


## 5. 선택 실습 — 사전학습 텍스트 모델

이 셀은 교재의 `transformers.pipeline("text-generation")` 흐름을 경량 모델로 체험하기 위한 선택 실습입니다.

- 인터넷 연결이 필요합니다.
- 첫 실행 시 모델을 다운로드합니다.
- 출력은 영어 중심이며 사실성이 보장되지 않습니다.
- 핵심 실습을 먼저 완료한 뒤 실행합니다.


In [ ]:
if RUN_OPTIONAL_PRETRAINED_DEMO:
    import subprocess
    import importlib.util

    if importlib.util.find_spec("transformers") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"])

    from transformers import pipeline, set_seed

    set_seed(SEED)
    hf_device = 0 if DEVICE == "cuda" else -1
    generator = pipeline("text-generation", model="distilgpt2", device=hf_device)
    prompt = "Generative AI helps data scientists"
    outputs = generator(
        prompt,
        max_new_tokens=45,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        num_return_sequences=2,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    for i, item in enumerate(outputs, 1):
        print(f"[{i}] {item['generated_text']}\n")
else:
    print("선택 실습이 꺼져 있습니다. RUN_OPTIONAL_PRETRAINED_DEMO=True로 바꾸면 실행됩니다.")


### 선택 비교

사전학습 모델을 실행했다면 다음을 비교하십시오.

- seed는 그대로 두고 temperature만 `0.4`, `0.8`, `1.3`으로 변경
- temperature는 그대로 두고 seed를 3개로 변경
- 결과를 **일관성, 다양성, 사실성 위험**의 세 열로 정리


## 6. 책임 있는 AI 미니 Audit

다음 표의 `나의 판단`과 `근거·조치`를 채우십시오. 실습 결과가 단순한 교육용 생성물이라도, 실제 모델을 사용할 때 적용할 습관을 연습합니다.


In [ ]:
audit_df = pd.DataFrame([
    {"점검축": "개인정보·동의", "질문": "실제 인물·대화·과제·음성을 허가 없이 사용했는가?", "나의 판단": "", "근거·조치": ""},
    {"점검축": "편향·공정성", "질문": "특정 집단에 대한 고정관념이나 배제가 나타나는가?", "나의 판단": "", "근거·조치": ""},
    {"점검축": "사실성·안전", "질문": "그럴듯하지만 검증되지 않은 내용을 사실처럼 제시하는가?", "나의 판단": "", "근거·조치": ""},
    {"점검축": "저작권·출처", "질문": "모델·데이터·생성물의 라이선스와 출처를 확인했는가?", "나의 판단": "", "근거·조치": ""},
    {"점검축": "책임·감독", "질문": "최종 판단과 수정 책임자가 명확한가?", "나의 판단": "", "근거·조치": ""},
])
display(audit_df)


### 실습 6-1. 사례 분석

아래 하나를 골라 5문장으로 작성하십시오.

- 실제 학생과 닮은 AI 인물을 학과 홍보물에 사용
- 존재하지 않는 장학금 조건을 챗봇이 생성
- 특정 직업 이미지가 한 성별로만 반복 생성
- 유명 가수와 비슷한 음성으로 홍보 노래 생성

반드시 포함할 내용: 이해관계자, 잠재 피해, 검증 방법, 공개 방법, 최종 책임자.


## 7. 미니 챌린지 — 생성 실험 보고서 한 장

아래 조건으로 텍스트·이미지·오디오 중 하나를 선택해 실험합니다.

1. prompt 1개를 정합니다.
2. seed 3개 또는 temperature 3개를 비교합니다.
3. 결과를 표로 정리합니다.
4. 가장 좋은 결과 하나를 고르고 기준을 설명합니다.
5. 실패 사례 또는 한계 하나를 반드시 기록합니다.
6. 책임 있는 AI 점검축 중 관련된 두 가지를 적용합니다.

**제출에 포함할 정보**

- 이름, 실행 날짜, Python·주요 패키지 버전
- prompt, seed, temperature 또는 기타 생성 파라미터
- 결과 3개와 비교표
- 5문장 해석
- 한계와 책임 있는 사용 점검


In [ ]:
# 미니 챌린지 기록 템플릿
experiment_log = {
    "student": STUDENT_NAME,
    "date": "YYYY-MM-DD",
    "modality": "text / image / audio",
    "prompt": "",
    "seeds": [SEED, SEED + 1, SEED + 2],
    "temperature_or_control": "",
    "selection_criterion": "",
    "observed_limitation": "",
    "responsible_ai_checks": [],
}

experiment_log


## 8. 퀴즈

### 객관식

1. 조건부 생성 모델을 가장 잘 나타내는 표현은?  
   A. `p(y | x)`  B. `p(x | c)`  C. `argmax(y)`  D. `MSE(x, y)`

2. seed의 주된 목적은?  
   A. 모델의 정확도를 자동으로 높인다.  
   B. 입력 데이터를 암호화한다.  
   C. 확률적 실험을 가능한 한 재현하고 비교하게 한다.  
   D. GPU 메모리를 줄인다.

3. 텍스트 생성에서 temperature를 크게 높였을 때 일반적으로 기대되는 변화는?  
   A. 다양성이 줄어든다.  B. 희귀 후보가 선택될 가능성이 커진다.  C. 사실성이 보장된다.  D. seed가 필요 없어지게 된다.

4. 2초 오디오를 16,000 Hz로 표현할 때 샘플 수는?  
   A. 8,000  B. 16,000  C. 32,000  D. 160,000

5. 생성 결과가 자연스럽고 유창하다는 사실만으로 결론 내릴 수 없는 것은?  
   A. 문법적 자연스러움  B. 사실성  C. 문장의 길이  D. 출력 형식

### 주관식

6. 예측 모델과 생성 모델의 출력 구조 차이를 한 문장으로 설명하십시오.

7. 실제 인물과 비슷한 이미지나 목소리를 생성할 때 최소 두 가지 점검사항을 쓰십시오.

8. 첫 실습에서 거대한 이미지·오디오 모델 대신 가벼운 기준선을 사용한 교육적 이유를 설명하십시오.


## 9. Take-home messages

- 생성은 학습된 분포에서 가능한 결과를 샘플링하는 과정입니다.
- prompt는 조건을, seed는 무작위성의 시작점을, temperature는 선택 분포의 날카로움을 조절합니다.
- 텍스트·이미지·오디오는 데이터 표현은 다르지만 공통 인터페이스를 공유합니다.
- 출력의 유창함·아름다움은 사실성·공정성·안전성을 보장하지 않습니다.
- 재현 가능한 실험을 위해 모델 ID, 버전, prompt, seed, 생성 파라미터와 실행환경을 기록합니다.
- 최종 판단과 사용 책임은 사람에게 있습니다.

### 참고자료

- 『핸즈온 생성형 AI』 1장 “생성 미디어 입문”
- 교재 제공 `01_introduction.ipynb`
- https://github.com/yk-genai/genaibook
- 향후 강의 저장소: https://github.com/lunalab-ai/genAI

### 다음 수업 예고

다음 수업에서는 언어 모델의 토큰화, 다음 토큰 예측, Transformer 블록과 어텐션의 기본 아이디어를 다룹니다.
